# Hill Climbing aplicado a la generación de horarios
## Fundamentos de Inteligencia Artificial

### Caso real
Se imparte la misma materia de Inteligencia Artificial a cuatro grupos distintos:

- LITIID-001
- LITIID-002
- LITIID-003
- LITIID-004

Cada grupo debe recibir **6 horas por semana** distribuidas obligatoriamente como:

**2 + 2 + 1 + 1**

Esto significa que cada grupo tendrá dos sesiones de 2 horas, dos sesiones de 1 hora y un día sin clase.

Además:

1. Las dos sesiones de 2 horas no pueden quedar en días consecutivos.
2. Se busca que la carga total diaria quede lo más balanceada posible entre lunes y viernes.

El objetivo de Hill Climbing será encontrar un horario válido y equilibrado.


## 1. Librería necesaria


In [ ]:
import random


## 2. Datos del problema

Cada grupo debe conservar siempre la estructura `[2, 2, 1, 1, 0]`. El orden puede cambiar entre los días.


In [ ]:
grupos = [
    "LITIID-001",
    "LITIID-002",
    "LITIID-003",
    "LITIID-004"
]

dias = [
    "Lunes",
    "Martes",
    "Miércoles",
    "Jueves",
    "Viernes"
]

distribucion_base = [2, 2, 1, 1, 0]


## 3. Generar un horario inicial

Cada grupo recibe una permutación aleatoria de `[2, 2, 1, 1, 0]`.

Así siempre se cumplen automáticamente las 6 horas semanales y la estructura 2+2+1+1.


In [ ]:
def generar_horario():

    horario = {}

    for grupo in grupos:

        # Copiar la distribución base
        horas = distribucion_base.copy()

        # Reordenar aleatoriamente los días
        random.shuffle(horas)

        # Guardar la distribución del grupo
        horario[grupo] = horas

    return horario


## 4. Contar violaciones

No se permite que un grupo tenga dos sesiones de 2 horas en días consecutivos.


In [ ]:
def contar_violaciones(horario):

    violaciones = 0

    for grupo in grupos:

        horas = horario[grupo]

        # Comparar:
        # Lunes-Martes
        # Martes-Miércoles
        # Miércoles-Jueves
        # Jueves-Viernes
        for i in range(4):

            if horas[i] == 2 and horas[i + 1] == 2:
                violaciones += 1

    return violaciones


## 5. Calcular horas totales por día

Hay 4 grupos de 6 horas:

`4 × 6 = 24 horas`

El promedio ideal es:

`24 / 5 = 4.8 horas por día`


In [ ]:
def horas_por_dia(horario):

    totales = [0, 0, 0, 0, 0]

    for grupo in grupos:

        for i in range(5):

            totales[i] += horario[grupo][i]

    return totales


## 6. Medir el desbalance diario

Un horario estará mejor balanceado cuando las horas diarias estén cerca del promedio de 4.8.


In [ ]:
def calcular_desbalance(horario):

    totales = horas_por_dia(horario)

    promedio = 24 / 5

    desbalance = 0

    for horas in totales:

        desbalance += abs(horas - promedio)

    return desbalance


## 7. Función de evaluación

Se utilizará:

`Costo = 100 × violaciones + desbalance`

Las violaciones reciben un peso alto porque primero queremos cumplir las restricciones y después mejorar el balance.

Como estamos minimizando:

**menor costo = mejor horario**


In [ ]:
def evaluar(horario):

    violaciones = contar_violaciones(horario)

    desbalance = calcular_desbalance(horario)

    costo = 100 * violaciones + desbalance

    return costo


## 8. Generar una solución vecina

Un vecino se obtiene haciendo un pequeño cambio:

1. elegir un grupo;
2. elegir dos días;
3. intercambiar sus horas.

Así siempre se conserva la estructura 2+2+1+1+0.


In [ ]:
def generar_vecino(horario):

    # Crear una copia del horario actual
    vecino = {
        grupo: horas.copy()
        for grupo, horas in horario.items()
    }

    # Elegir un grupo al azar
    grupo = random.choice(grupos)

    # Elegir dos días distintos
    dia1, dia2 = random.sample(range(5), 2)

    # Intercambiar las horas
    vecino[grupo][dia1], vecino[grupo][dia2] = (
        vecino[grupo][dia2],
        vecino[grupo][dia1]
    )

    return vecino


## 9. Algoritmo Hill Climbing

La lógica es:

Solución actual → generar vecinos → evaluar → elegir el mejor → moverse si mejora → detenerse si ya no mejora.


In [ ]:
def hill_climbing(max_iteraciones=100, numero_vecinos=20):

    # 1. Crear solución inicial
    actual = generar_horario()

    # 2. Evaluar solución inicial
    costo_actual = evaluar(actual)

    # 3. Repetir búsqueda de mejoras
    for iteracion in range(max_iteraciones):

        vecinos = []

        # 4. Generar varios vecinos
        for _ in range(numero_vecinos):

            vecino = generar_vecino(actual)

            vecinos.append(vecino)

        # 5. Suponer inicialmente que la mejor solución
        #    sigue siendo la actual
        mejor_vecino = actual
        mejor_costo = costo_actual

        # 6. Evaluar todos los vecinos
        for vecino in vecinos:

            costo = evaluar(vecino)

            # Como estamos minimizando,
            # un costo menor significa una solución mejor
            if costo < mejor_costo:

                mejor_vecino = vecino
                mejor_costo = costo

        # 7. Si ningún vecino mejora la solución actual,
        #    Hill Climbing termina
        if mejor_costo >= costo_actual:

            break

        # 8. Si hubo mejora, moverse al mejor vecino
        actual = mejor_vecino
        costo_actual = mejor_costo

    # 9. Devolver la mejor solución encontrada
    return actual, costo_actual


## 10. Mostrar el horario


In [ ]:
def mostrar_horario(horario):

    print(f"{'Grupo':<15}", end="")

    for dia in dias:
        print(f"{dia:<12}", end="")

    print()
    print("-" * 75)

    for grupo in grupos:

        print(f"{grupo:<15}", end="")

        for horas in horario[grupo]:

            print(f"{horas:<12}", end="")

        print()

    print("-" * 75)

    totales = horas_por_dia(horario)

    print(f"{'TOTAL':<15}", end="")

    for total in totales:

        print(f"{total:<12}", end="")

    print()


## 11. Ejecutar Hill Climbing

La semilla permite reproducir el mismo ejemplo durante la clase.


In [ ]:
random.seed(10)

mejor_horario, costo = hill_climbing()

mostrar_horario(mejor_horario)

print()
print("Violaciones:",
      contar_violaciones(mejor_horario))

print("Desbalance:",
      calcular_desbalance(mejor_horario))

print("Costo total:",
      costo)


## 12. Interpretación

Un buen resultado debería tener:

- `Violaciones = 0`
- una distribución diaria cercana a 4.8 horas;
- un costo total bajo.

Hill Climbing no prueba todas las combinaciones posibles. Parte de un horario, genera pequeñas modificaciones y se mueve únicamente cuando encuentra una solución mejor.
